# 7조 최종 모델 — 암환자 유전체 변이 기반 암종 분류

**최종 Public 리더보드 0.4501 (1위) · 그룹 5-fold OOF 0.4697 · SEED 42**

| 단계 | 구성 | 기여 |
|---|---|---|
| 특성 | 유전자 0/1 + 변이 유형·아미노산 치환 20×20·6그룹 전이·경로 부담 (5,281열) | LB 0.345~0.354 |
| 트리 | XGBoost 3구성(전체·규제·유전자300) × 시드 3개 = 9모델 확률 평균 | 0.3544 |
| 텍스트 | 변이 토큰(gene/mut/event) TF-IDF → LinearSVC | — |
| 블렌드 | 트리 0.6 + 텍스트 0.4 (가중치는 OOF로만 선택) | 0.3619 |
| 교정 ① | train·test 동일 변이 프로필 환자 → 형제 코호트 라벨(KIPAN↔KIRC·GBMLGG↔LGG) | **0.4480** |
| 교정 ② | 혼동쌍 전담 분류기 (LUAD↔LUSC·HNSC↔LUSC, OOF 검증 후 채택) | **0.4501** |

- 데이터 경로: `/data` (대회 규정). 외부 데이터 없음 · test.csv는 예측과 형식 처리(빈칸→WT)에만 사용 (fit/EDA 없음)
- 실행 시간: Colab CPU 기준 특성 생성 ~15분 + 학습 ~40분. 빠른 동작 확인은 아래 `FAST=True`
- 재현: 모든 난수 SEED=42 고정 (앙상블 시드 42/7/2026은 명시적 인자)

In [ ]:
# ========== 0. Config ==========
import os, re, sys, hashlib, warnings
import numpy as np, pandas as pd
import sklearn, scipy, xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
from scipy.special import softmax
warnings.filterwarnings('ignore')

SEED = 42
ENSEMBLE_SEEDS = (42, 7, 2026)
FAST = False          # True면 나무 수·시드를 줄여 몇 분 안에 끝까지 동작만 확인 (제출용 아님)

CANDIDATES = ['/data', 'data', '../data', '/content/drive/MyDrive/dacon_cancer/data']
DATA = next((p for p in CANDIDATES if os.path.exists(os.path.join(p, 'train.csv'))), None)
assert DATA, f'train.csv를 찾지 못했습니다. 확인한 경로: {CANDIDATES}'
print('data dir :', DATA)
print('python   :', sys.version.split()[0])
for m in (np, pd, sklearn, scipy, xgb): print(f'{m.__name__:<8}:', m.__version__)

실행 환경과 데이터 위치를 고정합니다. `/data`가 기본이고, 로컬·Colab 경로를 차례로 탐색합니다. 버전 출력은 재현성 확인용입니다.

In [ ]:
# ========== 1. Load ==========
train = pd.read_csv(os.path.join(DATA, 'train.csv'))
test  = pd.read_csv(os.path.join(DATA, 'test.csv'))
sub   = pd.read_csv(os.path.join(DATA, 'sample_submission.csv'))

GENES = [c for c in train.columns if c not in ('ID', 'SUBCLASS')]

def fill_missing(df):
    # test.csv에만 빈칸 237개(환자 128, 유전자 25) 존재. 정보 없음 = WT 로 처리 (형식 처리이며 분포는 보지 않음)
    out = df.copy(); out[GENES] = out[GENES].fillna('WT'); return out

train = fill_missing(train); test = fill_missing(test)
le = LabelEncoder(); y = le.fit_transform(train['SUBCLASS']); CLS = list(le.classes_); C = len(CLS)

def profile_hash(df):
    # 변이 프로필이 완전히 같은 환자는 같은 그룹 (교차검증 분할 & 교정 ①에 사용)
    return df[GENES].apply(lambda r: hashlib.md5('|'.join(r.values).encode()).hexdigest(), axis=1).values

grp_train = profile_hash(train)
print(train.shape, test.shape, '| classes:', C)

데이터를 읽고 ① test 빈칸을 WT로 채우고 ② 라벨 인코딩 ③ 변이 프로필 해시(동일 프로필 = 동일 그룹)를 만듭니다.

In [ ]:
# ========== 2. 구조 특성 (5,281열) ==========
AA_CODES = list("ACDEFGHIKLMNPQRSTVWY")
AA_INDEX = {aa: i for i, aa in enumerate(AA_CODES)}
AA_GROUPS = {"aliphatic": set("AVILM"), "aromatic": set("FWY"), "polar": set("STNQC"), "acidic": set("DE"), "basic": set("KRH"), "special": set("GP")}
AA_TO_GROUP = {aa: group for group, members in AA_GROUPS.items() for aa in members}
AA_CHANGE_RE = re.compile(r"(?<![A-Z])([ACDEFGHIKLMNPQRSTVWY])(\d+)([ACDEFGHIKLMNPQRSTVWY*])", re.I)
FS_RE = re.compile(r"fs", re.I)
INDEL_RE = re.compile(r"del|ins|dup", re.I)
WT_TOKENS = {"WT", "WILDTYPE", "WILD TYPE", "NORMAL"}
MISSING_TOKENS = {"", "NA", "N/A", "NAN", "NONE", "NULL", "<NA>"}

# 사전 지식으로 고정한 패널이며, 입력 행렬에 실제로 존재하는 유전자만 집계합니다.
PATHWAY_GENES = {
    "rtk_mapk": {"EGFR", "ERBB2", "ERBB3", "ERBB4", "FGFR1", "FGFR2", "FGFR3", "FGFR4", "MET", "ALK", "RET", "NTRK1", "NTRK2", "NTRK3", "KIT", "PDGFRA", "PDGFRB", "BRAF", "NF1", "PTPN11", "RAF1", "MAP2K1", "MAP2K2", "SOS1"},
    "pi3k_akt_mtor": {"PIK3CA", "PIK3CB", "PIK3R1", "PIK3R2", "AKT1", "AKT2", "AKT3", "PTEN", "MTOR", "TSC1", "TSC2", "RICTOR", "RPTOR"},
    "dna_repair": {"BRCA1", "BRCA2", "PALB2", "RAD51", "ATR", "CHEK1", "CHEK2", "FANCA", "FANCC", "FANCD2", "FANCG", "MLH1", "MSH2", "MSH6", "PMS2", "POLE", "POLD1", "ERCC2", "ERCC4"},
    "chromatin": {"ARID1B", "PBRM1", "SMARCA4", "SMARCB1", "KMT2A", "KMT2B", "KMT2C", "KMT2D", "SETD2", "EZH2", "EP300", "CREBBP", "DNMT3A", "TET2", "ASXL1"},
    "wnt": {"APC", "CTNNB1", "AXIN1", "AXIN2", "RNF43", "AMER1", "TCF7L2"},
    "tgf_beta": {"TGFBR1", "TGFBR2", "SMAD2", "SMAD3", "ACVR1B"},
    "cell_cycle_p53": {"TP53", "RB1", "CDKN2A", "CCND1", "CDK4", "CDK6", "MDM2", "CCNE1"},
}

def build_mutation_features(frame, gene_columns):
    """행별 파생변수를 생성하며, 전체 데이터 통계나 정답 레이블은 사용하지 않습니다."""
    n_rows = len(frame)
    raw = frame.loc[:, gene_columns]
    presence = np.zeros((n_rows, len(gene_columns)), dtype=np.uint8)
    missing_cells = np.zeros(n_rows, dtype=np.int32)
    count_names = ("missense_count", "stop_gain_count", "frameshift_count", "indel_marker_count", "same_residue_count", "unparsed_mutation_cell_count")
    counts = {name: np.zeros(n_rows, dtype=np.int32) for name in count_names}
    pairs = np.zeros((n_rows, len(AA_CODES) ** 2), dtype=np.int32)
    group_names = list(AA_GROUPS)
    group_pairs = np.zeros((n_rows, len(group_names) ** 2), dtype=np.int32)

    for col_index, gene in enumerate(gene_columns):
        values = raw[gene].astype("string").str.strip()
        normalized = values.str.upper()
        missing = normalized.isna() | normalized.isin(MISSING_TOKENS)
        missing_cells += missing.to_numpy(dtype=np.int32)
        mutated = values.notna() & ~missing & ~normalized.isin(WT_TOKENS)
        row_positions = np.flatnonzero(mutated.to_numpy())
        presence[row_positions, col_index] = 1
        for row_pos in row_positions:
            event_found = False
            seen_tokens = set()
            for token in re.split(r"[\s,;|]+", str(values.iloc[row_pos]).upper()):
                token = token.strip()
                if not token or token in seen_tokens:
                    continue
                seen_tokens.add(token)
                fs_hits = len(FS_RE.findall(token))
                if fs_hits:
                    counts["frameshift_count"][row_pos] += fs_hits
                    event_found = True
                    continue
                changes = AA_CHANGE_RE.findall(token)
                if changes:
                    event_found = True
                    for ref_aa, _position, alt_aa in set(changes):
                        if alt_aa == "*":
                            counts["stop_gain_count"][row_pos] += 1
                        elif ref_aa == alt_aa:
                            # 전사체 또는 코딩 서열 정보가 없으므로 동의 변이라고 단정하지 않습니다.
                            counts["same_residue_count"][row_pos] += 1
                        else:
                            counts["missense_count"][row_pos] += 1
                            pair_idx = AA_INDEX[ref_aa] * len(AA_CODES) + AA_INDEX[alt_aa]
                            pairs[row_pos, pair_idx] += 1
                            g1 = group_names.index(AA_TO_GROUP[ref_aa])
                            g2 = group_names.index(AA_TO_GROUP[alt_aa])
                            group_pairs[row_pos, g1 * len(group_names) + g2] += 1
                    continue
                if INDEL_RE.search(token):
                    counts["indel_marker_count"][row_pos] += 1
                    event_found = True
            if not event_found:
                counts["unparsed_mutation_cell_count"][row_pos] += 1

    feature_values = {f"gene_mutated__{gene}": presence[:, i] for i, gene in enumerate(gene_columns)}
    feature_values["mutated_gene_count"] = presence.sum(axis=1)
    feature_values["missing_gene_cell_count"] = missing_cells
    feature_values["missing_gene_cell_fraction"] = missing_cells / max(len(gene_columns), 1)
    feature_values.update(counts)
    feature_values["truncating_marker_count"] = counts["stop_gain_count"] + counts["frameshift_count"]
    feature_values["missense_per_mutated_gene"] = counts["missense_count"] / np.maximum(presence.sum(axis=1), 1)

    # 단백질 수준의 아미노산 분포로 변이 패턴을 근사하지만, 염기 문맥 시그니처는 복원할 수 없습니다.
    denom = np.maximum(counts["missense_count"], 1)
    for ref_aa in AA_CODES:
        for alt_aa in AA_CODES:
            pair_idx = AA_INDEX[ref_aa] * len(AA_CODES) + AA_INDEX[alt_aa]
            pair_count = pairs[:, pair_idx]
            feature_values[f"aa_pair_count__{ref_aa}_{alt_aa}"] = pair_count
            feature_values[f"aa_pair_rate__{ref_aa}_{alt_aa}"] = pair_count / denom
    for i, source_group in enumerate(group_names):
        for j, target_group in enumerate(group_names):
            pair_count = group_pairs[:, i * len(group_names) + j]
            feature_values[f"aa_group_transition_count__{source_group}_to_{target_group}"] = pair_count
            feature_values[f"aa_group_transition_rate__{source_group}_to_{target_group}"] = pair_count / denom

    available_gene_set = set(gene_columns)
    gene_index = {gene: i for i, gene in enumerate(gene_columns)}
    for pathway, panel in PATHWAY_GENES.items():
        available = [gene for gene in panel if gene in available_gene_set]
        panel_indices = [gene_index[gene] for gene in available]
        hits = presence[:, panel_indices].sum(axis=1) if panel_indices else np.zeros(n_rows, dtype=np.int32)
        feature_values[f"pathway__{pathway}__mutated_gene_count"] = hits
        feature_values[f"pathway__{pathway}__fraction_available"] = hits / max(len(available), 1)
    return pd.DataFrame(feature_values, index=frame.index).astype(np.float32)


print('특성 생성 중 — 환자×유전자 셀을 전부 파싱하므로 수 분 걸립니다')
Xtr = build_mutation_features(train, GENES)
Xte = build_mutation_features(test, GENES)
GENE_COLS = [c for c in Xtr.columns if c.startswith('gene_mutated__')]
SUMM_COLS = [c for c in Xtr.columns if not c.startswith('gene_mutated__')]
print('features:', Xtr.shape, '| gene 0/1:', len(GENE_COLS), '| summary:', len(SUMM_COLS))

변이 문자열을 환자 단위 숫자 특성으로 변환합니다 — 유전자별 0/1, 변이 유형 개수(미스센스·정지·프레임시프트·인델), 아미노산 치환 20×20 쌍과 6그룹 전이(개수·비율), 신호경로 7패널 부담. **행 단위 계산만 사용**하므로 train/test 어느 쪽에도 fit이 없습니다.

In [ ]:
# ========== 3. 텍스트 표현: 변이 토큰 TF-IDF -> LinearSVC ==========
WT_SET = {'WT','WILDTYPE','WILD TYPE','NORMAL','NA','N/A','NAN','NONE','NULL'}

def documents(frame):
    # 한 환자 = 한 문서. BRAF 열의 V600E -> gene_BRAF / mut_V600E / event_BRAF_V600E
    out = []
    for row in frame[GENES].itertuples(index=False, name=None):
        terms = []
        for gene, v in zip(GENES, row):
            if not isinstance(v, str): continue
            v = v.strip()
            if not v or v.upper() in WT_SET: continue
            seen = set()
            for tok in re.split(r'[\s,;|]+', v.upper()):
                tok = tok.strip()
                if not tok or tok in seen: continue
                seen.add(tok)
                m = re.sub(r'[^A-Z0-9*]+', '_', tok).strip('_')
                if m: terms.extend((f'gene_{gene}', f'mut_{m}', f'event_{gene}_{m}'))
        out.append(' '.join(terms) if terms else 'no_mutation_token')
    return out

def text_probs(train_docs, y_fit, pred_docs):
    # 어휘(fit)는 학습 문서에서만 — 예측 문서는 transform만
    vec = TfidfVectorizer(min_df=2, max_features=250000, sublinear_tf=True)
    A = vec.fit_transform(train_docs); B = vec.transform(pred_docs)
    mdl = LinearSVC(C=0.5, class_weight='balanced', random_state=SEED, max_iter=5000).fit(A, y_fit)
    d = mdl.decision_function(B)
    al = np.zeros((B.shape[0], C))
    for i, c in enumerate(mdl.classes_): al[:, c] = d[:, i]
    return softmax(al, axis=1)

D_train = documents(train); D_test = documents(test)
print('documents ready')

같은 환자를 두 번째 표현(문자열 토큰의 희소 벡터)으로 봅니다. 트리는 숫자 조합을, 선형 모델은 `V600E` 같은 문자열 규칙을 배우므로 서로 다른 실수를 하고, 섞으면 상쇄됩니다.

In [ ]:
# ========== 4. 트리 앙상블 (3구성 x 3시드) + 텍스트 -> 확률 ==========
HP = {'A': dict(n_estimators=100, learning_rate=0.1, max_depth=6),
      'B': dict(n_estimators=300, learning_rate=0.05, max_depth=4, min_child_weight=5,
                subsample=0.8, colsample_bytree=0.6, reg_lambda=5.0)}
CONFIGS = [(None, 'A'), (None, 'B'), (300, 'B')]   # (유전자 상위 N | None=전체, 하이퍼파라미터)
seeds = (42,) if FAST else ENSEMBLE_SEEDS
if FAST: HP = {k: {**v, 'n_estimators': 20} for k, v in HP.items()}

def columns_for(topn, frame_fit):
    if topn is None: return list(Xtr.columns)
    top = frame_fit[GENE_COLS].sum().sort_values(ascending=False).head(topn).index
    return SUMM_COLS + list(top)

def xgb_model(hp, seed):
    return xgb.XGBClassifier(tree_method='hist', random_state=seed, n_jobs=-1,
                             eval_metric='mlogloss', **HP[hp])

tree_probs = []
for topn, hp in CONFIGS:
    cols = columns_for(topn, Xtr)          # 전체 train 기준 상위 유전자 (test 정보 미사용)
    for seed in seeds:
        m = xgb_model(hp, seed)
        m.fit(Xtr[cols], y, sample_weight=compute_sample_weight('balanced', y))
        tree_probs.append(m.predict_proba(Xte[cols]))
        print(f'tree fit: topn={topn} hp={hp} seed={seed}')
P_tree = np.mean(tree_probs, axis=0)
P_text = text_probs(D_train, y, D_test)
print('probs:', P_tree.shape, P_text.shape)

트리 9모델(특성 거칠기 3단계 × 시드 3개)의 확률을 평균해 시드 운과 특성 선택의 우연을 지웁니다. 클래스 가중치(balanced)로 38명 DLBC도 786명 BRCA와 같은 비중으로 학습합니다.

In [ ]:
# ========== 5. 블렌드: 트리 0.6 + 텍스트 0.4 ==========
W_TEXT = 0.4   # OOF 탐색으로 선택, 리더보드에서 w=0.5/0.6/0.8 모두 하락 확인
P = (1 - W_TEXT) * P_tree + W_TEXT * P_text
pred = np.array(CLS, dtype=object)[P.argmax(1)]
print('blend done | STES 비율:', round(float(np.mean(pred == 'STES')) * 100, 1), '%')

가중치 0.4는 내부 OOF로 고르고 리더보드로 이웃 값(0.5·0.6·0.8 전부 하락)을 확인한 값입니다.

In [ ]:
# ========== 6. 교정 ①: 형제 코호트 라벨 ==========
# 근거(train 내부에서 검증): 변이 >=3개인 동일 프로필 그룹 422개가 전부 '같은 환자, 다른 코호트 라벨'.
# 따라서 test에서 train과 프로필이 일치하는 환자는 쪼개진 쌍의 반대쪽 복사본이며,
# 정답은 train 라벨의 형제 코호트다 (리더보드 검증: 0.3619 -> 0.4480).
SIBLING = {'KIPAN': 'KIRC', 'KIRC': 'KIPAN', 'GBMLGG': 'LGG', 'LGG': 'GBMLGG'}

h_tr = grp_train
h_te = profile_hash(test)
label_sets = pd.DataFrame({'h': h_tr, 'y': train['SUBCLASS']}).groupby('h')['y'].agg(set)
unique_label = {h: next(iter(s)) for h, s in label_sets.items() if len(s) == 1}
n_mut_te = Xte['mutated_gene_count'].to_numpy()

sib = np.array([SIBLING.get(unique_label.get(h)) for h in h_te], dtype=object)
matched = (sib != None) & (n_mut_te >= 1)
changed = int((pred[matched] != sib[matched]).sum())
pred[matched] = sib[matched]
print(f'프로필 일치 교정: {int(matched.sum())}명 (예측 변경 {changed}명)')

이 단계가 단일 최대 개선(+0.086)입니다. test 통계는 쓰지 않습니다 — train만으로 만든 조회표와 test **행 단위** 대조(정확 일치 1-NN과 동일)입니다.

In [ ]:
# ========== 7. 교정 ②: 혼동쌍 전담 분류기 ==========
# OOF에서 채택된 2쌍만 적용 (LUAD-LUSC +0.0009, HNSC-LUSC +0.0001 / OV-UCEC, COAD-STES는 기각)
top2 = np.argsort(-P, axis=1)[:, :2]
for a_name, b_name in [('LUAD', 'LUSC'), ('HNSC', 'LUSC')]:
    ia, ib = CLS.index(a_name), CLS.index(b_name)
    cand = np.array([set(t) == {ia, ib} for t in top2]) & ~matched
    sel = np.isin(y, [ia, ib]); yy = (y[sel] == ib).astype(int)
    mdl = xgb.XGBClassifier(n_estimators=20 if FAST else 200, max_depth=4, learning_rate=0.07,
                            tree_method='hist', random_state=SEED, n_jobs=-1, eval_metric='logloss',
                            subsample=0.8, colsample_bytree=0.6, reg_lambda=5.0)
    mdl.fit(Xtr[sel], yy, sample_weight=compute_sample_weight('balanced', yy))
    p = mdl.predict_proba(Xte[cand])[:, 1]
    pred[cand] = np.where(p > 0.5, b_name, a_name)
    print(f'{a_name}-{b_name} 전담: 후보 {int(cand.sum())}명')

확률 상위 2개가 정확히 그 쌍인 환자만, 그 두 클래스 환자로만 학습한 이진 모델이 다시 판정합니다. 4쌍을 OOF로 시험해 오르는 2쌍만 남겼습니다.

In [ ]:
# ========== 8. 제출 파일 ==========
out = sub.copy(); out['SUBCLASS'] = pred
assert list(out.columns) == ['ID', 'SUBCLASS'] and len(out) == len(test) and out['SUBCLASS'].isin(CLS).all()
out.to_csv('submission.csv', index=False)
print('saved submission.csv |', len(out), 'rows')
print(out['SUBCLASS'].value_counts().head(8))

---
## 검증 기록 (제출 로그와 동일 — `logs/experiments.md`)

| 날짜 | 구성 | 내부(OOF/CV) | Public LB |
|---|---|---|---|
| 11.06 | 우리 E8 특성 XGB | 0.450 | 0.307 |
| 11.09 | 본 특성 + 규제 XGB (명석) | 0.437* | 0.354 |
| 11.09 | 트리 9모델 평균 | 0.4636 | 0.3544 |
| 11.09 | + 텍스트 0.4 블렌드 | 0.4687 | 0.3619 |
| 11.10 | + 형제 라벨 교정 | — | **0.4480** |
| 11.10 | + 혼동쌍 전담·조회 확장 | 0.4697 | **0.4501 (1위)** |

\* 홀드아웃. 전체 과정·기각된 실험 포함 상세는 팀 레포 `logs/experiments.md`, `docs/team_guide.html` 참조.
규정 준수: 외부 데이터 미사용 · test fit/EDA 없음 · 어휘·유전자 선택·가중치 전부 train(또는 OOF)에서만 결정.